In [ ]:
from blueprint import MatrixPipeline, EloModifier
import pandas as pd
from constants import TEAM_ID_MAP, BEST_ALPHA, BEST_BETA
from helpers import (
    country_to_elo,
    safe_scrape_elo,
    match_outcome_probs_to_odds,
    calculate_market_rmse,
)
from simulator import run_pytorch_monte_carlo

global_q = pd.read_csv("./data/global_q_matrix.csv")

home_team = "Spain"
away_team = "Belgium"

elo_df = safe_scrape_elo()
elo_home = country_to_elo(elo_df, home_team)
elo_away = country_to_elo(elo_df, away_team)

ctx = {
    "home_team": home_team,
    "home_id": TEAM_ID_MAP[home_team],
    "away_team": away_team,
    "away_id": TEAM_ID_MAP[away_team],
    "elo_home": elo_home,
    "elo_away": elo_away,
    "alpha": BEST_ALPHA,
    "beta": BEST_BETA,
}
bookie_odds = [1.70, 4.00, 5.96]


baseline_pipeline = MatrixPipeline([EloModifier()])

q_grid = baseline_pipeline.build_grid(global_q, ctx)
prob_h, prob_d, prob_a = run_pytorch_monte_carlo(q_grid)
odds_h, odds_d, odds_a = match_outcome_probs_to_odds(prob_h, prob_d, prob_a)

print(f"MODEL ODDS: {odds_h} | {odds_d} | {odds_a}")
model_probs = [prob_h, prob_d, prob_a]
rmse = calculate_market_rmse(model_probs, bookie_odds)
print(f"RMSE: {rmse}")

In [ ]:
from blueprint import MatrixPipeline, EloModifier
import pandas as pd
from helpers import (
    safe_scrape_elo,
    match_outcome_probs_to_odds,
    calculate_market_rmse,
    matchup_to_ctx_dict,
)
from simulator import run_pytorch_monte_carlo

global_q = pd.read_csv("./data/global_q_matrix.csv")

home_team = "England"
away_team = "Norway"

# home_team = "Norway"
# away_team = "England"

elo_df = safe_scrape_elo()
ctx = matchup_to_ctx_dict(home=home_team, away=away_team, elo_df=elo_df)

# bookie_odds = [4.20, 3.80, 1.95]
bookie_odds = [1.95, 3.80, 4.20]


baseline_pipeline = MatrixPipeline([EloModifier()])

q_grid = baseline_pipeline.build_grid(global_q, ctx)
prob_h, prob_d, prob_a = run_pytorch_monte_carlo(q_grid)
odds_h, odds_d, odds_a = match_outcome_probs_to_odds(prob_h, prob_d, prob_a)

print(f"MODEL ODDS: {odds_h} | {odds_d} | {odds_a}")
model_probs = [prob_h, prob_d, prob_a]
rmse = calculate_market_rmse(model_probs, bookie_odds)
print(f"RMSE: {rmse}")